> 📄 English: [contextual_tutorial.ipynb](contextual_tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial)
<img src="https://github.com/ContextualAI/examples/blob/main/images/Contextual_AI_Lockup_Dark.png?raw=true" alt="Image description" width="160" />

 # 使用 Contextual AI 构建 RAG Agent 🚀
  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-RAG-with-Contextual/contextual_tutorial.ipynb)
 ## 为你的文档简化 RAG Agent 创建

 为企业应用构建检索增强生成（RAG）系统可能很复杂，涉及基础设施、分块和检索策略。本教程演示如何使用 **Contextual AI 的托管平台**来简化 RAG agent 的构建过程，让你专注于自己的用例，而不是底层细节。

 ### 为什么这很重要

 传统的 RAG 实现面临诸多挑战：
 - **复杂的基础设施**：搭建向量数据库、嵌入模型和检索系统
 - **文档处理**：处理多种格式、表格、图表和层级内容
 - **幻觉控制**：确保回复以源文档为依据
 - **性能评估**：测试并优化检索准确性和回复质量

 Contextual AI 提供了一个托管系统，旨在解决这些挑战，帮助你快速上手 RAG agent。

## 你将构建什么

在这个动手教程中，你将创建一个**用于金融文档分析和定量推理的完整 RAG agent**。

### 学习成果
完成本教程后，你将理解如何：
1. **创建和配置 datastore**，用于文档存储和索引
2. **摄入多种文档类型**，进行准确且层级感知的解析
3. **构建和部署 RAG agent**，附带自定义指令和安全保障
4. **通过自然语言查询并与你的 agent 交互**
5. **使用自动化测试框架评估和优化** agent 性能

⏱️ 本教程可在 15 分钟内运行完成。所有这些步骤也可以通过 GUI 完成，从而实现无代码的 RAG agent。

<div align="center">
<img src="https://github.com/user-attachments/assets/a0d8da57-b6b4-49d3-8a5a-4549b73b201c" alt="Contextual Architecture" width="1000"/>
</div>

---

## 环境配置

首先，我们安装所需依赖并配置开发环境。`contextual-client` 库为 Contextual AI 平台提供 Python 绑定，而其他包则支持数据可视化和进度跟踪。

In [ ]:
# Install required packages for Contextual AI integration and data visualization
%pip install contextual-client matplotlib tqdm requests pandas dotenv

接下来，我们导入整个教程中会用到的一些必要库：

In [ ]:
import os
import json
import requests
from pathlib import Path
from typing import List, Optional, Dict
from IPython.display import display, JSON
import pandas as pd
from contextual import ContextualAI
import ast

---

##  步骤 1：API 认证配置

### 获取你的 Contextual AI API 密钥

在开始构建 RAG agent 之前，你需要先访问 Contextual AI 平台。


### 分步 API 密钥配置：

1. **创建账户**：访问 [app.contextual.ai](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-app&target=https%3A%2F%2Fapp.contextual.ai%2F%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=app.contextual.ai) 并点击 **"Start Free"** 按钮
2. **进入 API Keys**：登录后，在侧边栏中找到 **"API Keys"**
3. **生成新密钥**：点击 **"Create API Key"** 并按照配置步骤操作
4. **安全存储**：复制你的 API 密钥并妥善保存（之后你将无法再次查看）

<div align="center">
<img src="https://github.com/ContextualAI/examples/blob/main/images/API_Keys.png?raw=true" alt="API" width="800"/>


</div>

### 配置你的 API 密钥

要运行本教程，你可以把 API 密钥存储在一个 `.env` 文件中。这样可以让密钥与代码分离。配置好 .env 文件后，你可以从 `.env` 加载 API 密钥来初始化 Contextual AI 客户端。如果你在 Google Colab 中，也可以使用 Google Secrets。

现在，你可以从 `.env` 加载 API 密钥来初始化 Contextual AI 客户端。

In [ ]:
# Load API key from .env or google secrets
from dotenv import load_dotenv
import os
try:
    # Try Colab secrets if in Google Colab
    from google.colab import userdata
    API_KEY = userdata.get('CONTEXTUAL_API_KEY')
except:
    # Fallback to environment variable
    load_dotenv()
    API_KEY = os.getenv('CONTEXTUAL_API_KEY')

if not API_KEY:
    raise ValueError("Please set your CONTEXTUAL_API_KEY in Colab Secrets or as an environment variable")

from contextual import ContextualAI
client = ContextualAI(api_key=API_KEY)

---

##  步骤 2：创建你的文档 Datastore

### 理解 Datastore

Contextual AI 中的 **datastore** 是一个安全、隔离的容器，用于存放你的文档及其处理后的表示。每个 datastore 提供：

- **🔒 隔离存储**：每个用例的文档都被分开且安全地保存
- **🧠 智能处理**：对上传的文档自动解析、分块和索引
- **⚡ 优化检索**：高性能的搜索和排序能力


### 为什么要分开使用多个 Datastore？

每个 agent 都应该有自己的 datastore，以确保：
- 不同用例之间的**数据隔离**
- 敏感文档集合的**安全合规**
- **性能优化**，agent 可以针对特定文档类型和查询模式进行定制

让我们为我们的金融文档分析 agent 创建一个 datastore：

In [ ]:
datastore_name = 'Financial_Demo_RAG'

# Check if datastore exists
datastores = client.datastores.list()
existing_datastore = next((ds for ds in datastores if ds.name == datastore_name), None)

if existing_datastore:
    datastore_id = existing_datastore.id
    print(f"Using existing datastore with ID: {datastore_id}")
else:
    result = client.datastores.create(name=datastore_name)
    datastore_id = result.id
    print(f"Created new datastore with ID: {datastore_id}")

---

##  步骤 3：文档摄入与处理

既然你的 agent 的 datastore 已经配置好，让我们往里面添加一些金融文档。Contextual AI 的文档处理引擎提供**企业级解析**，能够熟练处理：

- **📊 复杂表格**：金融数据、电子表格和结构化信息
- **📈 图表**：视觉化数据提取与解读
- **📑 多页文档**：带层级结构的长报告

### 支持的文件格式

该平台支持多种文档格式：
- **PDF**：PDF 文档
- **HTML**：网页和 HTML 文档
- **DOC/DOCX**：Microsoft Word 文档
- **PPT/PPTX**：PowerPoint 演示文稿

### 示例金融文档

本教程中，我们将使用一些示例金融文档来演示各种具有挑战性的场景：
- **季度营收报告**，跨多个文件包含复杂的表格信息
- **数据分析文档**，包含对比数据
- **统计报告**，包含图表和指标

让我们准备文档集合：

In [ ]:
import os
import requests

# Create data directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# File list with corresponding GitHub URLs
files_to_upload = [
    # NVIDIA quarterly revnue 24/25
    ("A_Rev_by_Mkt_Qtrly_Trend_Q425.pdf", "https://raw.githubusercontent.com/ContextualAI/examples/refs/heads/main/08-ai-workshop/data/A_Rev_by_Mkt_Qtrly_Trend_Q425.pdf"),
    # NVIDIA quarterly revenue 22/23
    ("B_Q423-Qtrly-Revenue-by-Market-slide.pdf", "https://raw.githubusercontent.com/ContextualAI/examples/refs/heads/main/08-ai-workshop/data/B_Q423-Qtrly-Revenue-by-Market-slide.pdf"),
    # Spurious correlations report - fun example of graphs and statistical analysis
    ("C_Neptune.pdf", "https://raw.githubusercontent.com/ContextualAI/examples/refs/heads/main/08-ai-workshop/data/C_Neptune.pdf"),
    # Another spurious correlations report - fun example of graphs and statistical analysis
    ("D_Unilever.pdf", "https://raw.githubusercontent.com/ContextualAI/examples/refs/heads/main/08-ai-workshop/data/D_Unilever.pdf")
]

### 文档下载与摄入流程

下面的单元格将会：
1. 从 Contextual AI 的示例仓库**下载文档**（如果尚未缓存）
2. **上传到 Contextual AI** 进行智能处理
3. **跟踪处理状态**和文档 ID，供后续参考

In [ ]:
# Download and ingest all files
document_ids = []
for filename, url in files_to_upload:
    file_path = f'data/{filename}'

    # Download file if it doesn't exist
    if not os.path.exists(file_path):
        print(f"Fetching {file_path}")
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for bad status codes
            with open(file_path, 'wb') as f:
                f.write(response.content)
        except Exception as e:
            print(f"Error downloading {filename}: {str(e)}")
            continue

    # Upload to datastore
    try:
        with open(file_path, 'rb') as f:
            ingestion_result = client.datastores.documents.ingest(datastore_id, file=f)
            document_id = ingestion_result.id
            document_ids.append(document_id)
            print(f"Successfully uploaded {filename} to datastore {datastore_id}")
    except Exception as e:
        print(f"Error uploading {filename}: {str(e)}")

print(f"Successfully uploaded {len(document_ids)} files to datastore")
print(f"Document IDs: {document_ids}")

### 检查文档

让我们在 [https://app.contextual.ai/](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-app&target=https%3A%2F%2Fapp.contextual.ai%2F%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=https%3A%2F%2Fapp.contextual.ai%2F) 上查看我们的文档

1. 导航到你的工作区
2. 在左侧菜单中选择 **Datastores**
3. 选择 **Documents**
4. 点击 **Inspect**（文档加载完成后）

你将看到正在进行的 datastore 上传：

摄入完成后，你可以查看文档列表、查看它们的元数据，也可以通过 API 删除文档。

**注意：** 文档的摄入和处理可能需要几分钟。如果文档仍在摄入中，你会看到 `status='processing'`。一旦摄入完成，状态将显示为 `status='completed'`。

你可以在 [此处](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-docs&target=https%3A%2F%2Fdocs.contextual.ai%2Fapi-reference%2Fdatastores-documents%2Fget-document-metadata%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=here) 了解更多关于元数据的信息。

In [ ]:
metadata = client.datastores.documents.metadata(datastore_id = datastore_id, document_id = document_ids[0])
print("Document metadata:", metadata)

---

##  步骤 4：Agent 创建与配置

现在，你将创建我们的 RAG agent，它会与你刚摄入的文档进行交互。

你可以使用以下附加参数自定义 Agent：

- **`system_prompt`** 用于你的 RAG 系统在生成回复时参考的指令。注意，这是截至 8.18.25 的默认提示词。
- **`suggested_queries`** 是一项用户体验特性，用于为 agent 预填查询，让新用户可以看到有趣的示例。

可选：你也可以在 [app.contextual.ai](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-app&target=https%3A%2F%2Fapp.contextual.ai%2F%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=app.contextual.ai) 的 UI 中配置或编辑你的 agent，试试把生成模型换成另一个 LLM！

你可以在 [此处](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-docs&target=https%3A%2F%2Fdocs.contextual.ai%2Fapi-reference%2Fagents%2Fcreate-agent%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=here) 找到所有附加参数。

In [ ]:
system_prompt = '''
You are a helpful AI assistant created by Contextual AI to answer questions about relevant documentation provided to you. Your responses should be precise, accurate, and sourced exclusively from the provided information. Please follow these guidelines:
* Only use information from the provided documentation. Avoid opinions, speculation, or assumptions.
* Use the exact terminology and descriptions found in the provided content.
* Keep answers concise and relevant to the user's question.
* Use acronyms and abbreviations exactly as they appear in the documentation or query.
* Apply markdown if your response includes lists, tables, or code.
* Directly answer the question, then STOP. Avoid additional explanations unless specifically relevant.
* If the information is irrelevant, simply respond that you don't have relevant documentation and do not provide additional comments or suggestions. Ignore anything that cannot be used to directly answer this query.
'''

agent_name = "Demo"

# Get list of existing agents
agents = client.agents.list()

# Check if agent already exists
existing_agent = next((agent for agent in agents if agent.name == agent_name), None)

if existing_agent:
    agent_id = existing_agent.id
    print(f"Using existing agent with ID: {agent_id}")
else:
    print("Creating new agent")
    app_response = client.agents.create(
        name=agent_name,
        description="Helpful Grounded AI Assistant",
        datastore_ids=[datastore_id],
        agent_configs={
        "global_config": {
            "enable_multi_turn": False # Turning this off for deterministic responses for this demo
        }
        },
        suggested_queries=[
            "What was NVIDIA's annual revenue by fiscal year 2022 to 2025?",
            "When did NVIDIA's data center revenue overtake gaming revenue?",
            "What's the correlation between the distance between Neptune and the Sun and Burglary rates in the US?",
            "What's the correlation between Global revenue generated by Unilever Group and Google searches for 'lost my wallet'?",
            "Does this imply that Unilever Group's revenue is derived from lost wallets?",
            "What's the correlation between the distance between Neptune and the Sun and Global revenue generated by Unilever Group?"
        ]
    )
    agent_id = app_response.id
    print(f"Agent ID created: {agent_id}")

### 在平台上查看我们的 Agent

访问：[https://app.contextual.ai/](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-app&target=https%3A%2F%2Fapp.contextual.ai%2F%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=https%3A%2F%2Fapp.contextual.ai%2F)

1. 导航到你的工作区
2. 从左侧菜单中选择 **Agents**
3. 选择你的 Agent
4. 尝试一个建议查询，或输入你的问题

---

##  步骤 5：查询 Agent

### 测试你的 RAG Agent

既然我们的 agent 已配置好并连接到金融文档，让我们用各种类型的查询来测试它的能力。

必需字段如下：

- **`agent_id`**：你的 Agent 的唯一标识符
- **`messages`**：构成用户查询的一条或多条消息列表

可选信息包括 `stream` 和 `conversation_id` 参数。你可以参考 [此处](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-docs&target=https%3A%2F%2Fdocs.contextual.ai%2Fapi-reference%2Fagents-query%2Fquery&text=here) 了解更多信息。

让我们试试这个查询：**"What was NVIDIA's annual revenue by fiscal year 2022 to 2025?"** 查询：

In [ ]:
query_result = client.agents.query.create(
    agent_id=agent_id,
    messages=[{
        "content": "What was NVIDIA's annual revenue by fiscal year 2022 to 2025?",
        "role": "user"
    }]
)
print(query_result.message.content)

你还可以从查询结果中访问更多信息。例如，你可以显示检索到的文档。

In [ ]:
import base64
import io
from PIL import Image
import matplotlib.pyplot as plt

def display_base64_image(base64_string, title="Document"):
    # Decode base64 string
    img_data = base64.b64decode(base64_string)

    # Create PIL Image object
    img = Image.open(io.BytesIO(img_data))

    # Display using matplotlib
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title)
    plt.show()

    return img

# Retrieve and display all referenced documents
for i, retrieval_content in enumerate(query_result.retrieval_contents):
    print(f"\n--- Processing Document {i+1} ---")

    # Get retrieval info for this document
    ret_result = client.agents.query.retrieval_info(
        message_id=query_result.message_id,
        agent_id=agent_id,
        content_ids=[retrieval_content.content_id]
    )

    print(f"Retrieval Info for Document {i+1}:")

    # Display the document image
    if ret_result.content_metadatas and ret_result.content_metadatas[0].page_img:
        base64_string = ret_result.content_metadatas[0].page_img
        img = display_base64_image(base64_string, f"Document {i+1}")
    else:
        print(f"No image available for Document {i+1}")

print(f"\nTotal documents processed: {len(query_result.retrieval_contents)}")

### Agent 测试的关键观察

我们的 RAG agent 展示了几项重要能力：

✅ **准确的数据提取**：找到具体的数值数据，并提供带引用的精确数字

✅ **来源标注**：包含内联引用，把回复链接回源文档


---

##  步骤 6：评估你的 Agent

### 理解 LMUnit 测试框架

为确保我们的 RAG agent 在生产中可靠运行，我们需要在手动测试之外进行系统化评估。Contextual AI 提供了 **LMUnit**——一个自然语言测试框架，从多个维度评估 RAG 系统。

更多详情，请查看 [blog post](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=contextual-lmunit&target=https%3A%2F%2Fcontextual.ai%2Fblog%2Flmunit%2F%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=blog%20post) 和 [full notebook](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-rag-with-contextual--contextual-tutorial&click=github-examples-blob&target=https%3A%2F%2Fgithub.com%2FContextualAI%2Fexamples%2Fblob%2Fmain%2F03-standalone-api%2F01-lmunit%2Flmunit.ipynb%3Futm_campaign%3Dagents-towards-production%26utm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_content%3Dnotebook&text=full%20notebook)。

### 为什么自动化评估很重要

仅靠手动测试对于生产级 RAG 系统来说是不够的。
**自然语言单元测试**使你能够：

- 将评估拆分为具体、可测试的标准
- 获取关于各种质量维度的细粒度反馈
- 推动 LLM 输出的系统化改进
- 支持特定领域的质量标准

在本示例中，我们将使用会跨所有回复运行的全局单元测试。以下是我们为定量推理选择的六个关键维度作为单元测试：

1. **准确性（Accuracy）**
   - *问题：* "Does the response accurately extract specific numerical data from the documents?"
   - *为什么：* 确保回复能正确识别和使用定量信息

2. **因果性（Causation）**
   - *问题：* "Does the agent properly distinguish between correlation and causation?"
   - *为什么：* 测试分析推理是否可靠，是否避免了逻辑谬误

3. **综合（Synthesis）**
   - *问题：* "Are multi-document comparisons performed correctly with accurate calculations?"
   - *为什么：* 验证跨来源综合信息的能力

4. **局限性（Limitations）**
   - *问题：* "Are potential limitations or uncertainties in the data clearly acknowledged?"
   - *为什么：* 确保适当的说明和透明度

5. **证据（Evidence）**
   - *问题：* "Are quantitative claims properly supported with specific evidence from the source documents?"
   - *为什么：* 测试结论是否恰当地基于源材料

6. **相关性（Relevance）**
   - *问题：* "Does the response avoid unnecessary information?"
   - *为什么：* 验证回复是否保持聚焦和简洁

以下是我们将要运行的单元测试，用于评估系统的表现：

In [ ]:
unit_tests = [
      "Does the response accurately extract specific numerical data from the documents?",
      "Does the agent properly distinguish between correlation and causation?",
      "Are multi-document comparisons performed correctly with accurate calculations?",
      "Are potential limitations or uncertainties in the data clearly acknowledged?",
      "Are quantitative claims properly supported with specific evidence from the source documents?",
      "Does the response avoid unnecessary information?"
]

LMUnit 是专门为评估自然语言单元测试而训练的，它提供：

* 连续的 1-5 分评分
* 跨不同标准的一致评估
* 优于通用 LLM 的表现
* 为评估添加评分标准（rubrics）的能力

让我们从一个简单的示例开始，了解 LMUnit 如何工作：

In [ ]:
response = client.lmunit.create(
                    query="What was NVIDIA's Data Center revenue in Q4 FY25?",
                   response = """NVIDIA's Data Center revenue for Q4 FY25 was $35,580 million.\[1\]()

                                This represents a significant increase from the previous quarter (Q3 FY25) when Data Center revenue was $30,771 million.[1]()

                                The full quarterly trend for Data Center revenue in FY25 was:
                                - Q4 FY25: $35,580 million
                                - Q3 FY25: $30,771 million
                                - Q2 FY25: $26,272 million
                                - Q1 FY25: $22,563 million[1]()
                              """,
                    unit_test="Does the response avoid unnecessary information?"
                )
print(response)

该回复**确实**包含了不必要的信息！稍后我们可以调整提示词，以更好地满足这一要求。

让我们先创建一个包含 6 个查询的评估数据集。该数据集是通过向我们的 RAG Agent 提出所列查询并保存生成的回复以用于评估而生成的。

In [ ]:
# RAG Evaluation Questions
queries = [
   "What was NVIDIA's Data Center revenue in Q4 FY25?",
   "What is the correlation coefficient between Neptune's distance from the Sun and US burglary rates?",
   "How did NVIDIA's total revenue change from Q1 FY22 to Q4 FY25?",
   "What are the four main reasons why spurious correlations work, according to the Tyler Vigen documents?",
   "Why should we be skeptical of the correlation between Unilever's revenue and Google searches for 'lost my wallet'?",
   "When did NVIDIA's data center revenue overtake gaming revenue?"
]

# Create DataFrame from questions
eval = pd.DataFrame({'prompt': queries})
eval['response'] = ''

# Query the agent for each question
for index, row in eval.iterrows():
   try:
       query_result = client.agents.query.create(
           agent_id=agent_id,
           messages=[{
               "content": row['prompt'],
               "role": "user"
           }]
       )
       eval.at[index, 'response'] = query_result.message.content
   except Exception as e:
       print(f"Error processing row {index}: {e}")
       eval.at[index, 'response'] = f"Error: {e}"

print(eval[['prompt', 'response']])
# prompt: save eval as csv

eval.to_csv('eval_input.csv', index=False)

现在让我们测试完整批次：

In [ ]:
def run_unit_tests_with_progress(
    df: pd.DataFrame,
    unit_tests: List[str],
    batch_size: int = 10
) -> List[Dict]:
    """
    Run unit tests with progress tracking and error handling.

    Args:
        df: DataFrame with prompt-response pairs
        unit_tests: List of unit test strings
        batch_size: Number of tests to run in parallel

    Returns:
        List of test results
    """
    results = []

    # Iterate over the DataFrame rows
    for idx in tqdm(range(0, len(df)), desc="Processing responses"):
        row = df.iloc[idx]
        row_results = []

        # Run each unit test on the current prompt-response pair
        for test in unit_tests:
            try:
                result = client.lmunit.create(
                    query=row['prompt'],
                    response=row['response'],
                    unit_test=test
                )

                # Collect the test result score and metadata if available
                row_results.append({
                    'test': test,
                    'score': result.score,
                    'metadata': result.metadata if hasattr(result, 'metadata') else None
                })

            except Exception as e:
                print(f"Error with prompt {idx}, test '{test}': {e}")
                row_results.append({
                    'test': test,
                    'score': None,
                    'error': str(e)
                })

        # Store results
        results.append({
            'prompt': row['prompt'],
            'response': row['response'],
            'test_results': row_results
        })

    return results

from tqdm import tqdm
results = run_unit_tests_with_progress(eval, unit_tests)

pd.DataFrame([(r['prompt'], r['response'], t['test'], t['score']) for r in results for t in r['test_results']], columns=['prompt', 'response', 'test', 'score']).to_csv(f"unit_test_results.csv", index=False)
df = pd.read_csv("unit_test_results.csv")

for result in results[:2]:  # Slice to get the first two entries
    print(f"\nPrompt: {result['prompt']}")
    print(f"Response: {result['response']}")
    print("Test Results:")
    for test_result in result['test_results']:
        print(f"- {test_result['test']}: {test_result['score']}")

现在，我们可以用极坐标图来可视化这些指标。极坐标图（也称为雷达图或蛛网图）特别适合可视化多维评分，因为它们擅长同时展示多个相关指标之间的模式和权衡。

In [ ]:
# For polar plots
import numpy as np
import matplotlib.pyplot as plt
from typing import Union, Tuple

#clustering analysis
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns

def map_test_to_category(test_question: str) -> str:
    """Map the full test question to its category."""
    category_mapping = {
        'Does the response accurately extract specific numerical data': 'ACCURACY',
        'Does the agent properly distinguish between correlation and causation': 'CAUSATION',
        'Are multi-document comparisons performed correctly': 'SYNTHESIS',
        'Are potential limitations or uncertainties in the data': 'LIMITATIONS',
        'Are quantitative claims properly supported with specific evidence': 'EVIDENCE',
        'Does the response avoid unnecessary information': 'RELEVANCE'
    }

    for key, value in category_mapping.items():
        if key.lower() in test_question.lower():
            return value
    return None

def create_unit_test_plots(results: List[Dict],
                          test_indices: Optional[Union[int, List[int]]] = None,
                          figsize: tuple = (10, 10)):
    """
    Create polar plot(s) for unit test results. Can plot either a single test,
    specific multiple tests, or all tests in a row.

    Args:
        results: List of dictionaries containing test results
        test_indices: Optional; Either:
            - None (plots all results)
            - int (plots single result)
            - List[int] (plots multiple specific results)
        figsize: Tuple specifying the figure size (width, height)
    """
    # Handle different input cases for test_indices
    if test_indices is None:
        indices_to_plot = list(range(len(results)))
    elif isinstance(test_indices, int):
        if test_indices >= len(results):
            raise IndexError(f"test_index {test_indices} is out of range. Only {len(results)} results available.")
        indices_to_plot = [test_indices]
    else:
        if not test_indices:
            raise ValueError("test_indices list cannot be empty")
        if max(test_indices) >= len(results):
            raise IndexError(f"test_index {max(test_indices)} is out of range. Only {len(results)} results available.")
        indices_to_plot = test_indices

    # Categories in desired order
    categories = ['ACCURACY', 'CAUSATION', 'SYNTHESIS',
                'LIMITATIONS', 'EVIDENCE', 'RELEVANCE']

    # Set up the angles for the polar plot
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))  # Close the plot

    # Calculate figure size based on number of plots
    num_plots = len(indices_to_plot)
    fig_width = figsize[0] * num_plots
    fig = plt.figure(figsize=(fig_width, figsize[1]))

    # Create a subplot for each result
    for plot_idx, result_idx in enumerate(indices_to_plot):
        result = results[result_idx]

        # Create subplot
        ax = plt.subplot(1, num_plots, plot_idx + 1, projection='polar')

        # Get scores for this result
        scores = []
        for category in categories:
            score = None
            for test_result in result['test_results']:
                mapped_category = map_test_to_category(test_result['test'])
                if mapped_category == category:
                    score = test_result['score']
                    break
            scores.append(score if score is not None else 0)

        # Close the scores array
        scores = np.concatenate((scores, [scores[0]]))

        # Plot the scores
        ax.plot(angles, scores, 'o-', linewidth=2)
        ax.fill(angles, scores, alpha=0.25)

        # Set the labels
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories)

        # Set the scale
        ax.set_ylim(0, 5)

        # Add grid
        ax.grid(True)

        # Add score values as annotations
        for angle, score, category in zip(angles[:-1], scores[:-1], categories):
            ax.text(angle, score + 0.2, f'{score:.2f}',
                    ha='center', va='bottom')

        # Add title for each subplot
        prompt = result['prompt']
        ax.set_title(f"Test {result_idx}\n{prompt}", pad=20)

    plt.tight_layout()
    return fig

# Plot the test results
fig = create_unit_test_plots(results, test_indices=[0, 1, 2])
fig = create_unit_test_plots(results, test_indices=[3, 4, 5])